---### Application numérique — Échantillons bootstrapAvec n = 5 observations {A,B,C,D,E}, trois échantillons bootstrap :| Bootstrap | Échantillon ||-----------|-------------|| 1 | A, A, B, C, E || 2 | B, C, D, D, E || 3 | A, C, E, E, E |Chaque échantillon contient ~63 % d'observations uniques.

In [ ]:
from sklearn.utils import resampleX = np.array(['A','B','C','D','E'])for b in range(3):    boot = resample(X)    print(f"Bootstrap {b+1}: {boot}")

**Code R correspondant :**```rx <- c("A", "B", "C", "D", "E")for (b in 1:3) {  boot <- sample(x, replace = TRUE)  cat(sprintf("Bootstrap %d: %s\n", b, paste(boot, collapse = " ")))}```

# STA211 — Analyse : Arbres de régression/classification & Méta-algorithmes
## Préparé pour le samedi 6 juin 2026
### Inspiré des cours de V. Audigier & N. Niang

Ce notebook explore :
1. **Arbres CART** — régression et classification
2. **Bagging** — Bootstrap Aggregating
3. **Forêts aléatoires** — Random Forests
4. **Boosting** — AdaBoost
5. **Comparaison** des performances

---
## 1. Chargement des données (German Credit)

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.tree import DecisionTreeClassifier, plot_treefrom sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifierfrom sklearn.metrics import roc_auc_score, accuracy_score, roc_curvefrom sklearn.datasets import fetch_openmlplt.rcParams['figure.dpi'] = 120# Chargementdata = fetch_openml(data_id=31, as_frame=True, parser='pandas')df = data.framedf.rename(columns={'class': 'Creditability'}, inplace=True)df['Creditability'] = (df['Creditability'] == 'good').astype(int)# Encodagedf_encoded = pd.get_dummies(df.drop(columns=['Creditability']), drop_first=True)df_encoded['Creditability'] = df['Creditability'].valuesX = df_encoded.drop(columns=['Creditability'])y = df_encoded['Creditability']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=1/3, random_state=235)print(f"Données d'apprentissage : {X_train.shape}")print(f"Données de test         : {X_test.shape}")print(f"Proportion de bons payeurs : {y_train.mean():.3f}")

---### Application numérique — AdaBoost pas à pasCinq clients, règle : score < 70 → +1 (Good). Poids initiaux = 0.20.| i | Score | y | f₁(x)=signe(70−score) ||---|---|---|---|| 1 | 62 | +1 | +1 ✓ || 2 | 85 | −1 | −1 ✓ || 3 | 72 | +1 | −1 ✗ || 4 | 65 | −1 | +1 ✗ || 5 | 78 | −1 | −1 ✓ |**Itération 1 :** e₁ = 0.40, α₁ = ½·log(1.5) = 0.203  Poids après mise à jour et normalisation : [0.167, 0.167, 0.250, 0.250, 0.167]**Itération 2 :** f₂(x) = signe(75−score), e₂ = 0.250, α₂ = 0.549**Classifieur final :** f̂(x) = signe(0.203·f₁(x) + 0.549·f₂(x))

In [ ]:
score = np.array([62, 85, 72, 65, 78])y     = np.array([1, -1, 1, -1, -1])n = len(score); w = np.ones(n) / nfor b in range(2):    pred = np.where(score < 70, 1, -1)    if b == 1: pred = np.where(score < 75, 1, -1)    e = w[pred != y].sum()    alpha = 0.5 * np.log((1 - e) / max(e, 1e-15))    w[pred != y] *= np.exp(alpha)    w[pred == y] *= np.exp(-alpha)    w /= w.sum()    print(f"B={b+1}: e={e:.4f}, alpha={alpha:.4f}, w={np.round(w,4)}")

**Code R correspondant :**```rscore <- c(62, 85, 72, 65, 78)y     <- c(1, -1, 1, -1, -1)n <- length(score); w <- rep(1/n, n)for (b in 1:2) {  if (b == 1) pred <- ifelse(score < 70, 1, -1)  if (b == 2) pred <- ifelse(score < 75, 1, -1)  e <- sum(w[pred != y])  alpha <- 0.5 * log((1 - e) / max(e, 1e-15))  w[pred != y] <- w[pred != y] * exp(alpha)  w[pred == y] <- w[pred == y] * exp(-alpha)  w <- w / sum(w)  cat(sprintf("B=%d: e=%.4f, alpha=%.4f\n", b, e, alpha))  cat("  w:", round(w, 4), "\n")}```

---
## 2. Arbre CART

In [ ]:
tree = DecisionTreeClassifier(random_state=235, min_samples_leaf=5)
tree.fit(X_train, y_train)

y_score_tree = tree.predict_proba(X_test)[:, 1]
y_pred_tree  = tree.predict(X_test)

auc_tree = roc_auc_score(y_test, y_score_tree)
err_tree = 1 - accuracy_score(y_test, y_pred_tree)

print(f"AUC CART    : {auc_tree:.4f}")
print(f"Erreur CART : {err_tree:.4f}")

---### Application numérique — Recherche du meilleur seuilSix clients, on cherche la division sur l'âge qui minimise MSE :| Client | Âge | Score ||--------|-----|-------|| 1 | 23 | 62 || 2 | 35 | 75 || 3 | 45 | 85 || 4 | 28 | 65 || 5 | 50 | 90 || 6 | 32 | 72 |**Scission à 30 ans :** ℛ₁={23,28}, ĉ₁=63.5 / ℛ₂={35,45,50,32}, ĉ₂=80.5  MSE = (62-63.5)²+(65-63.5)²+(75-80.5)²+(85-80.5)²+(90-80.5)²+(72-80.5)² = 217.5**Scission à 40 ans :** ℛ₁={23,35,28,32}, ĉ₁=68.5 / ℛ₂={45,50}, ĉ₂=87.5  MSE = (62-68.5)²+(75-68.5)²+(65-68.5)²+(72-68.5)²+(85-87.5)²+(90-87.5)² = 121.5→ Le seuil **40 ans** minimise MSE.

In [ ]:
import numpy as npage = np.array([23, 35, 45, 28, 50, 32])score = np.array([62, 75, 85, 65, 90, 72])for s in [30, 40]:    c1, c2 = score[age <= s].mean(), score[age > s].mean()    mse = ((score[age <= s] - c1)**2).sum() + ((score[age > s] - c2)**2).sum()    print(f"Seuil {s}: MSE = {mse:.2f}")

**Code R correspondant :**```rage <- c(23, 35, 45, 28, 50, 32)score <- c(62, 75, 85, 65, 90, 72)for (s in c(30, 40)) {  c1 <- mean(score[age <= s]); c2 <- mean(score[age > s])  mse <- sum((score[age <= s] - c1)^2) + sum((score[age > s] - c2)^2)  cat(sprintf("Seuil %d: MSE = %.2f", s, mse))}```

In [ ]:
# Visualisation de l'arbre (limité à profondeur 3 pour lisibilité)
plt.figure(figsize=(16, 8))
plot_tree(tree, max_depth=3, feature_names=X.columns.tolist(),
          class_names=['Mauvais', 'Bon'], filled=True, rounded=True,
          fontsize=9)
plt.title('Arbre CART (profondeur ≤ 3)')
plt.show()

---### Application numérique — Critères d'impuretéRégion avec 4 bons payeurs (Good) et 2 mauvais (Bad) :  p̂<sub>G</sub> = 4/6 = 0.667, p̂<sub>B</sub> = 2/6 = 0.333**Gini :** 0.667×0.333 + 0.333×0.667 = **0.444****Déviance :** −(0.667×log(0.667) + 0.333×log(0.333)) = **0.636**

In [ ]:
p = np.array([4/6, 2/6])gini = (p * (1 - p)).sum()dev  = -(p * np.log(p)).sum()print(f"Gini = {gini:.4f}, Deviance = {dev:.4f}")

**Code R correspondant :**```rp <- c(4/6, 2/6)gini <- sum(p * (1 - p))dev  <- -sum(p * log(p))cat(sprintf("Gini = %.4f, Deviance = %.4f", gini, dev))```

---
## 3. Bagging

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(min_samples_leaf=5, random_state=235),
    n_estimators=200, random_state=235, oob_score=True, bootstrap=True
)
bag.fit(X_train, y_train)

y_score_bag = bag.predict_proba(X_test)[:, 1]
auc_bag = roc_auc_score(y_test, y_score_bag)
err_bag = 1 - accuracy_score(y_test, bag.predict(X_test))

print(f"AUC Bagging    : {auc_bag:.4f}")
print(f"Erreur Bagging : {err_bag:.4f}")
print(f"Erreur OOB     : {1 - bag.oob_score_:.4f}")

---
## 4. Forêt aléatoire

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_features='sqrt',
    random_state=235, oob_score=True
)
rf.fit(X_train, y_train)

y_score_rf = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_score_rf)
err_rf = 1 - accuracy_score(y_test, rf.predict(X_test))

print(f"AUC Random Forest    : {auc_rf:.4f}")
print(f"Erreur Random Forest : {err_rf:.4f}")
print(f"Erreur OOB           : {1 - rf.oob_score_:.4f}")

In [ ]:
# Importance des variables
importances = pd.DataFrame({
    'variable': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 variables importantes (Mean Decrease Impurity) :")
display(importances.head(10))

# Graphique
top10 = importances.head(10)
plt.figure(figsize=(8, 5))
plt.barh(range(len(top10)), top10['importance'].values, color='steelblue')
plt.yticks(range(len(top10)), top10['variable'].values)
plt.xlabel('Importance')
plt.title('Top 10 variables importantes (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---### Application numérique — Importance par permutationMSE avant et après permutation de l'âge :| y | ŷ original | ŷ (âge permuté) ||---|---|---|| 62 | 65 | 68 || 75 | 78 | 78 || 85 | 82 | 82 || 65 | 68 | 65 || 90 | 88 | 88 || 72 | 70 | 72 |MSE<sub>base</sub> = 46/6 = 7.67  MSE<sub>perm</sub> = 54/6 = 9.00  **VI(Âge) = 9.00 − 7.67 = 1.33**

In [ ]:
from sklearn.metrics import mean_squared_errory = np.array([62, 75, 85, 65, 90, 72])y_pred = np.array([65, 78, 82, 68, 88, 70])y_perm = np.array([68, 78, 82, 65, 88, 72])vi = mean_squared_error(y, y_perm) - mean_squared_error(y, y_pred)print(f"Importance Age: {vi:.2f}")

**Code R correspondant :**```ry <- c(62, 75, 85, 65, 90, 72)y_pred <- c(65, 78, 82, 68, 88, 70)y_perm <- c(68, 78, 82, 65, 88, 72)vi <- mean((y - y_perm)^2) - mean((y - y_pred)^2)cat(sprintf("Importance Age: %.2f\n", vi))```

---
## 5. Boosting (AdaBoost avec stumps)

In [ ]:
boost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=235),
    n_estimators=500, learning_rate=1.0, random_state=235
)
boost.fit(X_train, y_train)

y_score_boost = boost.predict_proba(X_test)[:, 1]
auc_boost = roc_auc_score(y_test, y_score_boost)
err_boost = 1 - accuracy_score(y_test, boost.predict(X_test))

print(f"AUC Boosting    : {auc_boost:.4f}")
print(f"Erreur Boosting : {err_boost:.4f}")

---
## 6. Comparaison et visualisation

In [ ]:
resultats = pd.DataFrame({
    'Méthode': ['CART', 'Bagging', 'Forêt aléatoire', 'Boosting'],
    'AUC':     [auc_tree, auc_bag, auc_rf, auc_boost],
    'Erreur':  [err_tree, err_bag, err_rf, err_boost]
})

print("=== SYNTHÈSE DES RÉSULTATS ===")
display(resultats.round(4))

In [ ]:
plt.figure(figsize=(14, 5))

# Courbes ROC
plt.subplot(1, 2, 1)
for name, y_score in [
    ('CART', y_score_tree),
    ('Bagging', y_score_bag),
    ('Random Forest', y_score_rf),
    ('Boosting', y_score_boost)
]:
    fpr, tpr, _ = roc_curve(y_test, y_score)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, y_score):.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC')
plt.legend()
plt.grid(alpha=0.3)

# Barres comparatives
plt.subplot(1, 2, 2)
x = np.arange(len(resultats))
width = 0.35
plt.bar(x - width/2, resultats['AUC'].values, width, label='AUC', color='steelblue')
plt.bar(x + width/2, 1 - resultats['Erreur'].values, width, label='Précision', color='coral')
plt.xticks(x, resultats['Méthode'].values)
plt.ylabel('Score')
plt.title('Comparaison des méthodes')
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 7. Résumé

| Méthode | Biais | Variance | Parallélisable | Sur-apprentissage |
|---------|-------|----------|----------------|-------------------|
| CART | Faible | Élevée | — | Oui |
| Bagging | Faible | Réduite | Oui | Non |
| Forêt aléatoire | Faible | Très réduite | Oui | Non |
| Boosting | Très faible | Variable | Non | Possible |

**Points clés à retenir :**
- Les arbres CART sont simples et interprétables mais instables
- Le bagging réduit la variance en moyennant des modèles sur bootstrap
- Les forêts aléatoires ajoutent la randomisation des variables
- Le boosting réduit biais ET variance mais peut surapprendre
- L'importance des variables permet de retrouver de l'interprétabilité

---
*Références : Hastie, Tibshirani & Friedman (2009) ; Breiman (2001) ; Tufféry (2015)*